# Transformación del dataset (audio) en ventanas de n segundos con label

### Instalar e importar librerías necesarias

In [ ]:
!pip install selenium
!pip install pdfrw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 2.8 MB/s eta 0:00:00


In [ ]:
!pip install kaggle

In [ ]:
import librosa
import IPython.display as ipd
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import soundfile as sf
from numpy import pi
from scipy.fftpack import fft, fftfreq

### Importar dataset (cuando no está)



In [ ]:
from google.colab import files

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Descargar el dataset. Reemplaza 'israaelmorsy/lung-sound-classification' con el nombre de usuario/dataset correcto si es diferente.
!kaggle datasets download -d vbookshelf/respiratory-sound-database

Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
100% 3.68G/3.69G [00:42<00:00, 31.3MB/s]
100% 3.69G/3.69G [00:42<00:00, 94.2MB/s]


In [ ]:
!unzip respiratory-sound-database.zip

Archive:  respiratory-sound-database.zip
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Pr_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Pr_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/102_1b1_Ar_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/102_1b1_Ar_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/103_2b2_Ar_mc_LittC2SE.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/103_2b2_Ar_mc_LittC2SE.wav  
  inflating: Respiratory_Sound_

### Obtener los audios

In [ ]:
import os
import librosa

audio_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
audio_data_list = []

for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        file_path = os.path.join(audio_dir, filename)
        y, sr = librosa.load(file_path, sr=None) # Cargar los audios con librosa
        audio_data_list.append(y)

print(f"Carga de {len(audio_data_list)} archivos de audio.")

Carga de 920 archivos de audio.


In [124]:
# Leer un audio para hacerle las pruebas

audio_file_path = None
for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        audio_file_path = os.path.join(audio_dir, filename)
        break

if audio_file_path:
    y, sr = librosa.load(audio_file_path) # Se define el audio como una señal de onda y sr referente al sample rate
    print(f"Successfully loaded audio file: {audio_file_path}")
else:
    print("No .wav files found in the directory.")

Successfully loaded audio file: /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/200_3p4_Pr_mc_AKGC417L.wav


In [ ]:
y, sr = librosa.load("/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav")

In [125]:
Audio(y, rate=sr)
print(Audio(y,rate=sr))

<IPython.lib.display.Audio object>


In [126]:
print("Ventana 1:")
display(Audio(y, rate=sr))

Ventana 1:


In [127]:
#Estudie cada uno de los parámetros del objeto creado para leer el archivo de audio.
F= sr #sample rate o frecuencia del audio
dt = y.size / sr  #
T = 1 / sr #Periodo
t = np.r_[0:dt:T]
print(
    f'y[t] tiene {y.size} muestras', #Cantidad de muestras en el audio
    f'La frecuencia de muestreo es {sr} Hz',
    f'y(t) tiene {dt:.1f}s '#duracion
    , sep='\n')


y[t] tiene 441000 muestras
La frecuencia de muestreo es 22050 Hz
y(t) tiene 20.0s 


In [128]:
# Calcular el número de muestras por ventana de n segundos

n = 4  # Duración de la ventana en segundos (le puse 4)
samples_per_window = n * sr

# Crear ventanas de 4 segundos
audio_windows = []
for i in range(0, len(y), samples_per_window):
    window = y[i:i + samples_per_window]
    audio_windows.append(window)

print(f"El audio se ha dividido en {len(audio_windows)} ventanas de aproximadamente 4 segundos cada una.")

El audio se ha dividido en 5 ventanas de aproximadamente 4 segundos cada una.


In [129]:
# Escuchar el audio de 4 segundos (Ventana 1) original

print("Ventana 1:")
display(Audio(audio_windows[0], rate=sr))

Ventana 1:


### Lectura de .txt para el síntoma

In [130]:
txt_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
txt_files_list = []

for filename in os.listdir(txt_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(txt_dir, filename)
        txt_files_list.append(file_path)

print(f"Se hallaron {len(txt_files_list)} archivos .txt.")

Se hallaron 920 archivos .txt.


In [131]:
# Obtener el nombre del audio que se escogió arriba
audio_basename = os.path.splitext(os.path.basename(audio_file_path))[0]

# Hallar el .txt con el mismo nombre
corresponding_txt_file = None
for txt_file_path in txt_files_list:
    txt_basename = os.path.splitext(os.path.basename(txt_file_path))[0]
    if txt_basename == audio_basename:
        corresponding_txt_file = txt_file_path
        break

print (corresponding_txt_file)

/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/200_3p4_Pr_mc_AKGC417L.txt


In [159]:
corresponding_txt_file = "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.txt"

In [160]:
# Leer .txt y procesar
if corresponding_txt_file:
    print(f"Procesando {os.path.basename(corresponding_txt_file)}:")
    with open(corresponding_txt_file, 'r') as f:
        lines = f.readlines()
        txt_windows = 0
        inc = 0
        label_symp = []
        while txt_windows < len(lines):
            current_lines = []
            total_duration = 0
            symp_labels = []

            # Verificar que sea menor a 4
            for i in range(n):
                if txt_windows + i < len(lines):
                    line = lines[txt_windows + i].strip().split()
                    if len(line) >= 4:
                        try:
                            start_time = float(line[0])
                            end_time = float(line[1])
                            duration = end_time - start_time
                            current_lines.append(line)
                            total_duration += duration
                            symp_labels.append(line[-2:])

                            if total_duration >= 4:
                                break # Detenerse
                        except ValueError:
                            print(f"Saltando línea {txt_windows + i + 1}: no hay valor numérico")
                            continue # Saltar línea
                    else:
                        print(f"Saltando línea {txt_windows + i + 1}: menos de cuatro columnas")
                        continue # Saltar línea
                else:
                    break # No more lines to check

            if not current_lines:
                txt_windows += 1
                continue # Pasar de línea

            # Determina el síntoma (para label)
            final_symp = ['0', '0']
            for symp in symp_labels:
                if symp[0] == '1' or final_symp[0] == '1':
                    final_symp[0] = '1'
                if symp[1] == '1' or final_symp[1] == '1':
                    final_symp[1] = '1'

            label_symp.append(final_symp[0] + final_symp[1])

            # Mover con el número de líneas procesadas
            txt_windows += len(current_lines)

            print(f"Ventana procesada con {len(current_lines)} líneas. Duración total: {total_duration:.2f}s. Etiqueta: {final_symp[0]+final_symp[1]}")

else:
    print("No se encontró el archivo .txt correspondiente.")

Procesando 101_1b1_Al_sc_Meditron.txt:
Ventana procesada con 4 líneas. Duración total: 5.76s. Etiqueta: 00
Ventana procesada con 3 líneas. Duración total: 5.36s. Etiqueta: 00
Ventana procesada con 3 líneas. Duración total: 5.56s. Etiqueta: 00
Ventana procesada con 2 líneas. Duración total: 3.26s. Etiqueta: 00


In [161]:
print(label_symp)

['00', '00', '00', '00']


In [165]:
labeled_audio_windows = []

# Ajustar la longitud de label_symp para que coincida con audio_windows
while len(audio_windows) != len(label_symp):
    if len(audio_windows) < len(label_symp):
        label_symp.pop()  # Eliminar la última etiqueta si hay más etiquetas que ventanas
    else:
        label_symp.append(label_symp[-1]) # Duplicar la última etiqueta si hay menos etiquetas que ventanas

# Crear una lista de tuplas (ventana de audio, etiqueta)
for i, window in enumerate(audio_windows):
    # Usar la etiqueta correspondiente de la lista label_symp ajustada
    label = label_symp[i]
    labeled_audio_windows.append((window, label))

print(f"Se han creado {len(labeled_audio_windows)} ventanas de audio con labels.")
if labeled_audio_windows:
    print(labeled_audio_windows[0])
else:
    print("No se crearon ventanas de audio etiquetadas.")

Se han creado 5 ventanas de audio con labels.
(array([0.        , 0.        , 0.        , ..., 0.1814879 , 0.17334947,
       0.173633  ], dtype=float32), '00')


In [166]:
print(label_symp)

['00', '00', '00', '00', '00']


In [167]:
print(inc)
print(len(label_symp))
print(len(audio_windows))

0
5
5


### Conversión a tensor

In [137]:
# Esta conversión a tensor me toca revisarla mejor

import torch
import numpy as np

# Separar los datos de audio y las etiquetas

audio_data_list = [item[0] for item in labeled_audio_windows]
labels = label_symp

# Convertir la lista de arrays de numpy a una lista de tensores de torch
audio_tensors = [torch.from_numpy(audio).float() for audio in audio_data_list]

# Apilar los tensores para crear un único tensor, asumiendo que todos los tensores tienen el mismo tamaño después del padding
try:
    audio_tensor = torch.stack(audio_tensors)
    print("Datos de audio convertidos a tensor.")
    print(f"Forma del tensor de audio: {audio_tensor.shape}")
except RuntimeError as e:
    print(f"Error al apilar tensores: {e}")
    print("Esto puede ocurrir si las ventanas de audio tienen tamaños diferentes.")
    print("Considera rellenar o truncar a una longitud uniforme antes de apilar.")

# Convertir etiquetas a un tensor

unique_labels = list(set(labels))
label_mapping = {label: i for i, label in enumerate(unique_labels)}
encoded_labels = [label_mapping[label] for label in labels]
label_tensor = torch.tensor(encoded_labels)

print(f"Etiquetas (primeras 5): {labels[:5]}")
print(f"Etiquetas codificadas (primeras 5): {encoded_labels[:5]}")
print(f"Forma del tensor de etiquetas: {label_tensor.shape}")

Datos de audio convertidos a tensor.
Forma del tensor de audio: torch.Size([5, 88200])
Etiquetas (primeras 5): ['00', '00', '00', '00']
Etiquetas codificadas (primeras 5): [0, 0, 0, 0]
Forma del tensor de etiquetas: torch.Size([4])


In [ ]:
# Read the first element of the tensor
if 'audio_tensor' in locals() and audio_tensor.shape[0] > 0:
    print("First element of the audio tensor:")
    print(audio_tensor[0])
else:
    print("Audio tensor not found or is empty. Please run the cell that creates the audio tensor.")

First element of the audio tensor:
tensor([0.0000, 0.0000, 0.0000,  ..., 0.1815, 0.1733, 0.1736])


### Filtro pasabanda

In [168]:
from scipy.signal import butter, filtfilt

# Definir los parámetros
lowcut = 45.0  # Lower cutoff (Hz)
highcut = 55.0 # Upper cutoff (Hz)
order = 5      # Orden del filtro (no estoy seguro de cuánto debería ser)

# Implementación del filtro
nyquist = 0.5 * sr
low = lowcut / nyquist
high = highcut / nyquist
b, a = butter(order, [low, high], btype='band')

# Aplicar el filtro Butterworth
filtered_y = filtfilt(b, a, y)

print("Filtro pasabanda aplicado.")

Filtro pasabanda aplicado.


In [169]:
# Calcular el número de muestras por ventana de n segundos

n = 4  # Duración de la ventana en segundos (le puse 4)
samples_per_window = n * sr

# Crear ventanas de 4 segundos a partir del audio filtrado
audio_windowsF = []
for i in range(0, len(filtered_y), samples_per_window):
    windowF = filtered_y[i:i + samples_per_window]
    audio_windowsF.append(windowF)

print(f"El audio filtrado se ha dividido en {len(audio_windows)} ventanas de aproximadamente 4 segundos cada una.")

El audio filtrado se ha dividido en 5 ventanas de aproximadamente 4 segundos cada una.


In [170]:
# Escuchar el audio de 4 segundos (Ventana 1) filtrado

print("Ventana 1:")
display(Audio(audio_windowsF[0], rate=sr))

Ventana 1:


/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


In [171]:
# Descargar

import soundfile as sf
from google.colab import files

output_filename_filtered = 'primera_ventana_filtrada.wav'
sf.write(output_filename_filtered, audio_windowsF[0], sr)

files.download(output_filename_filtered)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>